# 21 · Randstorm positive control: uncompressed public keys, 2012-04 to 2014-03

Inputs: tables from 05; estimator of 20

Refines the Randstorm contrast of 20 (first seen 2011–15 vs 2016–19) toward the wallets described in the disclosure.

Source: Unciphered, *Randstorm: You Can't Patch a House of Cards* (2023-11-14): wallets generated with BitcoinJS (JSBN) until 2014-03, highest risk before 2012-03, mainly browser wallets.

- Bitcoin Core uses compressed public keys by default from v0.6 (2012-03); browser wallets of the period (BitcoinJS 0.1.3) used uncompressed keys.
- Treated U: group-L hash160 keys first seen 2012-04-01 to 2014-03-31 whose public key in the spending script is uncompressed (65 bytes, prefix `04`). Control C: same period and conditions, compressed (33 bytes). Both arms are in group L, where the key format is observable.
- Robustness window `wide`: 2012-04 to 2015-02 (Bitcore 0.1.6–0.9.5 used JSBN until 2015-02).
- Limitation: other software of the period also used uncompressed keys without being affected (Electrum 1.x, Armory, hand-made paper wallets), so the treated group is enriched for, not restricted to, vulnerable wallets, and the estimate is a diluted lower bound.

Estimation as in 20 (Randstorm and FTX use the pool-mean baseline). `RS2` (keys with an incoming transfer in the previous 12 months, as for placebo dates) is the main version; `RS2_all` drops the restriction. All dates are estimated.

In [ ]:
# Configuration
PROJECT_ID = "your-gcp-project"
DATASET    = "qbtc_raw"
LOCATION   = "US"
AUTO_RUN_USD, MAX_USD_PER_QUERY, PRICE_PER_TIB_USD = 5.0, 15.0, 6.25
CONFIRM = False
FORCE_REBUILD = False
FREEZE_DATE = "2026-09-22"

PUBFORM_FS = ("2011-01-01", "2016-12-31")   # format determined for this range (wider than the treatment window)
WINDOWS = {"main": ("2012-04-01", "2014-03-31"), "wide": ("2012-04-01", "2015-02-28")}
H_MAIN, BANDWIDTHS, POOLS = 90, [60, 90, 120], ["main", "sym30"]
REPORT_DAYS = (7, 30)
OUTS = ["y1_strict", "y1_strict_held", "y3", "y0"]
INPUTS = "bigquery-public-data.crypto_bitcoin.inputs"

try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass
from google.cloud import bigquery
import json, os, time, datetime as dt
import numpy as np
import pandas as pd

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
DS = f"{PROJECT_ID}.{DATASET}"
SPENT = {"usd": 0.0}

def dry(sql):
    job = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = job.total_bytes_processed / 2**30
    return gb, gb / 1024 * PRICE_PER_TIB_USD

def q(sql):
    gb, usd = dry(sql)
    assert usd < AUTO_RUN_USD, f"estimated ${usd:.2f}, above the auto-run limit"
    SPENT["usd"] += usd
    return client.query(sql).to_dataframe()

def table_rows(name):
    try:
        return client.get_table(f"{DS}.{name}").num_rows
    except Exception:
        return None

def build(name, sql, force=False):
    n = table_rows(name)
    if n and not (FORCE_REBUILD or force):
        print(f"[{name}] exists, {n:,} rows; skipped"); return
    gb, usd = dry(sql)
    if usd > MAX_USD_PER_QUERY:
        raise RuntimeError(f"[{name}] estimated ${usd:.2f}, above the per-query limit ${MAX_USD_PER_QUERY}")
    if usd > AUTO_RUN_USD and not CONFIRM:
        raise RuntimeError(f"[{name}] estimated {gb:,.0f} GB ~ ${usd:.2f}, above ${AUTO_RUN_USD}: set CONFIRM=True and re-run this cell")
    print(f"[{name}] estimated {gb:,.0f} GB ~ ${usd:.2f}")
    t0 = time.time()
    job = client.query(sql, job_config=bigquery.QueryJobConfig(maximum_bytes_billed=int(gb * 1.25 * 2**30) + 10 * 2**30))
    print(f"[{name}] job {job.job_id}")
    job.result()
    billed = (job.total_bytes_billed or 0) / 2**30
    SPENT["usd"] += billed / 1024 * PRICE_PER_TIB_USD
    print(f"[{name}] done, {billed:,.0f} GB, {time.time() - t0:.0f}s, {table_rows(name):,} rows")

for t in ["panel_keys", "panel_key_flows", "panel_hazard", "panel_event", "address_key", "address_stats",
          "key_flags", "placebo_draws_strict"]:
    client.get_table(f"{DS}.{t}")
print("Required tables found")

## Step 1 · Public-key format, `key_pubform`

Group-L hash160 keys first seen 2011–2016. A P2PKH unlocking script is ⟨signature⟩⟨public key⟩: last 132 hex characters `41 04…` (65-byte push) → uncompressed `U`; last 68 `21 02/03…` (33-byte push) → compressed `C`. "both" indicates a parsing error. Only the monthly partitions of `crypto_bitcoin.inputs` containing first spends of these addresses are scanned.

Check: uncompressed share by month of first appearance around Bitcoin Core 0.6 (2012-03).

In [ ]:
CAND = f'''
  SELECT DISTINCT pk.key_id
  FROM `{DS}.panel_keys` pk
  WHERE pk.grp = 'L' AND pk.family = 'hash160'
    AND pk.first_seen_date BETWEEN DATE '{PUBFORM_FS[0]}' AND DATE '{PUBFORM_FS[1]}'
'''
months = q(f'''
WITH cand AS ({CAND})
SELECT DISTINCT DATE_TRUNC(s.first_spend_date, MONTH) AS m
FROM `{DS}.address_key` k
JOIN cand USING (key_id)
JOIN `{DS}.address_stats` s ON s.address = k.address
WHERE STARTS_WITH(k.address, '1') AND s.first_spend_date IS NOT NULL
  AND s.first_spend_date <= DATE '{FREEZE_DATE}'
''')
MONTHS = sorted(pd.to_datetime(months.m))
# inputs is partitioned on block_timestamp: consecutive months merged into constant ranges for pruning
RANGES, a = [], None
for i_, m in enumerate(MONTHS):
    if a is None: a = m
    nxt = MONTHS[i_ + 1] if i_ + 1 < len(MONTHS) else None
    if nxt is None or nxt != m + pd.offsets.MonthBegin(1):
        RANGES.append((a, m + pd.offsets.MonthBegin(1))); a = None
TS_FILTER = " OR ".join(f"(i.block_timestamp >= TIMESTAMP '{x:%Y-%m-%d}' AND i.block_timestamp < TIMESTAMP '{y:%Y-%m-%d}')" for x, y in RANGES)
print(f"First spends of the candidate addresses fall in {len(MONTHS)} months, {len(RANGES)} consecutive ranges: {MONTHS[0]:%Y-%m} ... {MONTHS[-1]:%Y-%m}")

SQL_PF = f'''
CREATE OR REPLACE TABLE `{DS}.key_pubform` AS
WITH cand AS ({CAND}),
ad AS (
  SELECT k.address, k.key_id, s.first_spend_date
  FROM `{DS}.address_key` k
  JOIN cand USING (key_id)
  JOIN `{DS}.address_stats` s ON s.address = k.address
  WHERE STARTS_WITH(k.address, '1') AND s.first_spend_date IS NOT NULL
),
i AS (
  SELECT a AS address, i.script_hex
  FROM `{INPUTS}` i, UNNEST(i.addresses) AS a
  WHERE ({TS_FILTER})
    AND i.type = 'pubkeyhash'
),
f AS (
  SELECT ad.key_id,
    CASE WHEN LENGTH(i.script_hex) >= 132 AND SUBSTR(i.script_hex, LENGTH(i.script_hex) - 131, 4) = '4104' THEN 'U'
         WHEN LENGTH(i.script_hex) >= 68 AND SUBSTR(i.script_hex, LENGTH(i.script_hex) - 67, 2) = '21'
              AND SUBSTR(i.script_hex, LENGTH(i.script_hex) - 65, 2) IN ('02', '03') THEN 'C'
         ELSE 'X' END AS form
  FROM i JOIN ad USING (address)
)
SELECT key_id, COUNTIF(form = 'U') AS n_u, COUNTIF(form = 'C') AS n_c, COUNTIF(form = 'X') AS n_x,
  CASE WHEN COUNTIF(form = 'U') > 0 AND COUNTIF(form = 'C') = 0 THEN 'U'
       WHEN COUNTIF(form = 'C') > 0 AND COUNTIF(form = 'U') = 0 THEN 'C'
       WHEN COUNTIF(form = 'U') > 0 THEN 'mixed' ELSE 'unknown' END AS pubform
FROM f
GROUP BY key_id
'''
build("key_pubform", SQL_PF)

chk = q(f'''
WITH cand AS ({CAND})
SELECT IFNULL(pf.pubform, 'not_found') AS pubform, COUNT(*) AS n_keys
FROM cand LEFT JOIN `{DS}.key_pubform` pf USING (key_id)
GROUP BY 1 ORDER BY 2 DESC''')
display(chk)
assert chk.loc[chk.pubform == "mixed", "n_keys"].sum() <= 0.001 * chk.n_keys.sum(), "uncompressed and compressed on the same key: parsing error"

by_m = q(f'''
SELECT DATE_TRUNC(k.first_seen_date, MONTH) AS m, COUNTIF(pf.pubform = 'U') AS n_u, COUNTIF(pf.pubform = 'C') AS n_c
FROM `{DS}.key_pubform` pf JOIN `{DS}.key_flags` k USING (key_id)
GROUP BY 1 ORDER BY 1''')
by_m["m"] = pd.to_datetime(by_m.m)
by_m["share_u"] = by_m.n_u / (by_m.n_u + by_m.n_c)
by_m.to_csv("pubform_by_month.csv", index=False)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(by_m.m, by_m.share_u, marker=".")
for d, lab in [("2012-03-30", "Bitcoin Core 0.6"), ("2014-03-01", "BitcoinJS drops JSBN"), ("2015-02-28", "Bitcore 0.9.5")]:
    ax.axvline(pd.Timestamp(d), color="grey", ls=":"); ax.text(pd.Timestamp(d), 1.0, lab, fontsize=7, rotation=90, va="top")
for w, (a, b) in WINDOWS.items():
    ax.axvspan(pd.Timestamp(a), pd.Timestamp(b), alpha=0.08 if w == "wide" else 0.15, color="C3")
ax.set_ylabel("share uncompressed"); ax.set_title("L-group hash160 keys: uncompressed-pubkey share by first-seen month", fontsize=9)
plt.tight_layout(); plt.savefig("fig_21_pubform.png", dpi=130); plt.show()

## Step 2 · `panel_rs2`

Per event and window (main / wide): U and C keys by four CEM dimensions (balance, coin age, last incoming transfer, number of UTXOs; first-seen period and script family fixed) and `is_large_mover`; for r = 0…30: sums of cumulative shares (capped at 1), sums of squares, BTC-weighted sums, and keys with first de-risking (y1 strict ≥ 50%) by r.

In [ ]:
v_cols = ",\n    ".join(f"MAX(IF(f.rel_day <= r, f.vc_{o}, 0)) AS v_{o}" for o in OUTS)
agg = ",\n  ".join(f"SUM(IFNULL(f.v_{o}, 0)) AS c_{o}, SUM(POW(IFNULL(f.v_{o}, 0), 2)) AS c2_{o}, "
                   f"SUM(kd.sats * IFNULL(f.v_{o}, 0)) / 1e8 AS cb_{o}" for o in OUTS)
SQL_RS2 = f'''
CREATE OR REPLACE TABLE `{DS}.panel_rs2`
CLUSTER BY event
AS
WITH k AS (
  SELECT pk.event, pk.key_id, pk.sats, pk.bal_bin, pk.age_bin, pk.rec_bin, pk.nutxo_bin, pk.is_large_mover,
         pf.pubform AS arm,
         pk.first_seen_date BETWEEN DATE '{WINDOWS["main"][0]}' AND DATE '{WINDOWS["main"][1]}' AS in_main,
         pk.first_seen_date BETWEEN DATE '{WINDOWS["wide"][0]}' AND DATE '{WINDOWS["wide"][1]}' AS in_wide
  FROM `{DS}.panel_keys` pk
  JOIN `{DS}.key_pubform` pf ON pf.key_id = pk.key_id
  WHERE pk.grp = 'L' AND pk.family = 'hash160' AND pf.pubform IN ('U', 'C')
),
kw AS (
  SELECT k.*, w FROM k, UNNEST(ARRAY_CONCAT(IF(in_main, ['main'], []), IF(in_wide, ['wide'], []))) AS w
),
f AS (
  SELECT f.event, f.key_id, r,
    {v_cols}
  FROM `{DS}.panel_key_flows` f, UNNEST(GENERATE_ARRAY(0, 30)) AS r
  WHERE f.key_id IN (SELECT key_id FROM `{DS}.key_pubform` WHERE pubform IN ('U', 'C'))
  GROUP BY f.event, f.key_id, r
),
h AS (
  SELECT event, key_id, t_y1s_50 FROM `{DS}.panel_hazard`
  WHERE key_id IN (SELECT key_id FROM `{DS}.key_pubform` WHERE pubform IN ('U', 'C'))
),
kd AS (
  SELECT kw.*, r FROM kw, UNNEST(GENERATE_ARRAY(0, 30)) AS r
)
SELECT kd.event, kd.w AS win, kd.arm, CONCAT(kd.bal_bin, '|', kd.age_bin, '|', kd.rec_bin, '|', kd.nutxo_bin) AS stratum,
  kd.bal_bin, kd.rec_bin, kd.is_large_mover, kd.r AS rel_day,
  COUNT(*) AS n, SUM(kd.sats) / 1e8 AS bal,
  {agg},
  COUNTIF(h.t_y1s_50 <= kd.r) AS c_h50
FROM kd
LEFT JOIN f ON f.event = kd.event AND f.key_id = kd.key_id AND f.r = kd.r
LEFT JOIN h ON h.event = kd.event AND h.key_id = kd.key_id
GROUP BY kd.event, win, kd.arm, stratum, kd.bal_bin, kd.rec_bin, kd.is_large_mover, rel_day
'''
build("panel_rs2", SQL_RS2)
rs2 = q(f"SELECT * FROM `{DS}.panel_rs2`")
ev = q(f'''SELECT event, ANY_VALUE(event_set) AS event_set, ANY_VALUE(event_date) AS event_date
           FROM `{DS}.panel_event` WHERE rel_day = 0 GROUP BY event''')
ev["event_date"] = pd.to_datetime(ev.event_date)
ev = ev.sort_values("event_date").reset_index(drop=True)
sz = rs2[(rs2.rel_day == 0)].groupby(["win", "event", "arm"]).n.sum().unstack("arm")
display(sz.loc["main"].join(ev.set_index("event").event_set).sort_values("event_set").head(20))

## Step 3 · CEM difference θ (U − C), baseline and randomization inference as in 20

In [ ]:
def theta(df):
    # df: cell rows of one event and rel_day (including arm). Returns key-weighted / BTC-weighted U - C for each outcome
    g = df.groupby(["stratum", "arm"]).sum(numeric_only=True).unstack("arm").fillna(0)
    if ("n", "U") not in g or ("n", "C") not in g:
        return {}
    g = g[(g[("n", "U")] > 0) & (g[("n", "C")] > 0)]
    nU, nC, bU, bC = g[("n", "U")], g[("n", "C")], g[("bal", "U")], g[("bal", "C")]
    out = {"n_U_matched": nU.sum(), "n_C_matched": nC.sum()}
    for o in [f"c_{x}" for x in OUTS] + ["c_h50"]:
        cU, cC = g[(o, "U")], g[(o, "C")]
        out[f"{o}|key|diff"] = (cU.sum() - (nU / nC * cC).sum()) / nU.sum()
        out[f"{o}|key|T"] = cU.sum() / nU.sum()
        if o != "c_h50":
            ob = o.replace("c_", "cb_")
            out[f"{o}|btc|diff"] = (g[(ob, "U")].sum() - (bU / bC * g[(ob, "C")]).sum()) / bU.sum()
            out[f"{o}|btc|T"] = g[(ob, "U")].sum() / bU.sum()
    return out

recs = []
for (win, e, r), d in rs2.groupby(["win", "event", "rel_day"]):
    full = r in REPORT_DAYS                       # subgroups and no_lm only on reported days; paths need only all/all
    for contrast, dd in [("RS2", d[d.rec_bin != "365d+"]), ("RS2_all", d)]:
        for variant, ddd in [("all", dd), ("no_lm", dd[~dd.is_large_mover])][: 2 if full else 1]:
            sgs = [("all", ddd)] + ([(b, ddd[ddd.bal_bin == b]) for b in sorted(ddd.bal_bin.unique())] if full else [])
            for sg, dx in sgs:
                t = theta(dx)
                for k_, v in t.items():
                    if "|" in k_:
                        o, w, stat = k_.split("|")
                        recs.append((win, contrast, variant, sg, o, w, stat, r, e, v))
L = pd.DataFrame(recs, columns=["win", "contrast", "variant", "subgroup", "outcome", "weight", "stat", "rel_day", "event", "value"])
SER = ["win", "contrast", "variant", "subgroup", "outcome", "weight", "stat", "rel_day"]
TH = L.pivot_table(index=SER, columns="event", values="value", aggfunc="first").reindex(columns=ev.event.tolist())
print("Series:", len(TH))

In [ ]:
T0 = pd.Timestamp("2020-01-01")
tday = ((ev.event_date - T0).dt.days).to_numpy(float)
EV_IDX = {e: i for i, e in enumerate(ev.event)}
draws_all = q(f"SELECT pool, draw_id, slot, fake_date FROM `{DS}.placebo_draws_strict`")
draws_all["fake_date"] = pd.to_datetime(draws_all.fake_date)
pl = ev[ev.event_set == "placebo"].set_index("event_date").event
POOL_EVENTS, DRAWS = {}, {}
for pool in POOLS:
    d = draws_all[draws_all.pool == pool]
    dates = sorted(d.fake_date.unique())
    missing = [x for x in dates if x not in pl.index]
    assert not missing, f"dates of pool {pool} not found in panel_event: {missing}"
    POOL_EVENTS[pool] = [pl[x] for x in dates]
    pos = {x: j for j, x in enumerate(dates)}
    D = np.zeros((d.draw_id.nunique(), len(dates)))
    for r in d.itertuples():
        D[r.draw_id - 1, pos[r.fake_date]] += 1
    DRAWS[pool] = D / D.sum(1, keepdims=True)
    print(f"pool {pool}: {len(dates)} dates, {len(D)} draws")

def smoother(pool, h, mask=None):
    idx = np.array([EV_IDX[e] for e in POOL_EVENTS[pool]])
    tp = tday[idx]
    ok = np.ones(len(idx), bool) if mask is None else mask
    lo = tp[ok].min() - 60
    S = np.zeros((len(ev), len(idx)))
    for i in range(len(ev)):
        w = np.exp(-0.5 * ((tp - tday[i]) / h) ** 2) * ok
        self_ = idx == i
        w[self_] = 0
        if tday[i] < lo:                           # before the pool: pool mean
            w = ok.astype(float); w[self_] = 0
            S[i] = w / w.sum(); continue
        X = np.column_stack([np.ones_like(tp), tp - tday[i]])
        A = X.T @ (X * w[:, None])
        S[i] = np.linalg.solve(A, (X * w[:, None]).T)[0]
    return S

def effects(Th, pool, h):
    idx = [EV_IDX[e] for e in POOL_EVENTS[pool]]
    Y = Th.to_numpy(float)
    P = Y[:, idx]
    B = np.full_like(Y, np.nan)
    nanmask = np.isnan(P)
    keys = pd.Series([m.tobytes() for m in nanmask])
    for key, rows_ in keys.groupby(keys).groups.items():
        rows_ = np.array(list(rows_))
        ok = ~nanmask[rows_[0]]
        if ok.sum() < 5:
            continue
        S = smoother(pool, h, ok)
        B[rows_] = Y[rows_] - np.nan_to_num(P[rows_]) @ S.T
    R = B[:, idx]
    return pd.DataFrame(B, index=Th.index, columns=Th.columns), R

def ri_p(beta, R):
    b = np.abs(np.asarray(beta, float))[:, :, None]
    r = np.abs(R)[:, None, :]
    cnt = np.sum(r >= b - 1e-15, axis=2)
    n = np.sum(~np.isnan(R), axis=1)[:, None]
    p = (1 + cnt) / (1 + n)
    p[np.isnan(np.asarray(beta, float))] = np.nan
    return p

def ri_ci(beta, R, a=0.10):
    lo_q, hi_q = np.nanquantile(R, 1 - a / 2, axis=1), np.nanquantile(R, a / 2, axis=1)
    return beta - lo_q[:, None], beta - hi_q[:, None]
print("Helpers ready")

In [ ]:
REPORT = ev[ev.event_set.isin(["main", "secondary", "nonquantum_news", "positive_control"])].event.tolist()
MAIN = ev[ev.event_set == "main"].event.tolist()
diff, lev = TH.xs("diff", level="stat"), TH.xs("T", level="stat")
out, pooled = [], []
for pool in POOLS:
    for h in BANDWIDTHS:
        B, R = effects(diff, pool, h)
        cols = [c for c in REPORT if c in B.columns]
        p = ri_p(B[cols].to_numpy(), R)
        lo, hi = ri_ci(B[cols].to_numpy(), R)
        for j, e in enumerate(cols):
            out.append(pd.DataFrame({"event": e, "pool": pool, "h": h, "theta": diff[e].values, "beta": B[e].values,
                                     "p_ri": p[:, j], "ci90_lo": lo[:, j], "ci90_hi": hi[:, j],
                                     "level_T": lev.reindex(diff.index)[e].values,
                                     "z_placebo_sd": B[e].values / np.nanstd(R, axis=1, ddof=1)}, index=diff.index).reset_index())
        mb = B[MAIN].mean(axis=1).to_numpy()
        null = np.nan_to_num(R) @ DRAWS[pool].T
        pp = (1 + np.sum(np.abs(null) >= np.abs(mb)[:, None], axis=1)) / (1 + null.shape[1])
        pooled.append(pd.DataFrame({"pool": pool, "h": h, "beta_mean": mb, "p_ri": pp,
                                    "ci90_lo": mb - np.quantile(null, .95, axis=1),
                                    "ci90_hi": mb - np.quantile(null, .05, axis=1)}, index=diff.index).reset_index())
        if pool == "main" and h == H_MAIN:
            B_MAIN, R_MAIN = B, R
eff = pd.concat(out, ignore_index=True).join(ev.set_index("event")[["event_set", "event_date"]], on="event")
pooled = pd.concat(pooled, ignore_index=True)
eff[eff.rel_day.isin(REPORT_DAYS)].to_csv("est_21_effects.csv", index=False)
pooled[pooled.rel_day.isin(REPORT_DAYS)].to_csv("est_21_pooled.csv", index=False)

s = eff[(eff.pool == "main") & (eff.h == H_MAIN) & (eff.variant == "all") & (eff.subgroup == "all") & eff.rel_day.isin(REPORT_DAYS)]
print("Positive controls (Randstorm / FTX):")
display(s[s.event.isin(["randstorm", "ftx"]) & (s.weight == "key")]
        .pivot_table(index=["win", "contrast", "event", "outcome"], columns="rel_day", values=["beta", "p_ri", "z_placebo_sd", "level_T"]).round(5))
print("Main events:")
ps = pooled[(pooled.pool == "main") & (pooled.h == H_MAIN) & (pooled.variant == "all") & (pooled.subgroup == "all") & pooled.rel_day.isin(REPORT_DAYS)]
display(ps[ps.weight == "key"].pivot_table(index=["win", "contrast", "outcome"], columns="rel_day", values=["beta_mean", "p_ri"]).round(5))

# figure: Randstorm beta(r) with the placebo band
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, o in zip(axes, ["c_y1_strict", "c_h50"]):
    for win, col in [("main", "C3"), ("wide", "C1")]:
        key_ = (win, "RS2_all", "all", "all", o, "key")
        if key_ not in B_MAIN.index.droplevel("rel_day"):
            continue
        b = B_MAIN.loc[key_]
        ax.plot(b.index, b["randstorm"], color=col, label=f"Randstorm, window {win}")
        rr = pd.DataFrame(R_MAIN, index=B_MAIN.index).loc[key_]
        ax.fill_between(b.index, rr.quantile(.05, axis=1), rr.quantile(.95, axis=1), color=col, alpha=0.12)
    ax.axhline(0, color="k", lw=0.6); ax.set_title(f"{o}: uncompressed − compressed (CEM), placebo 5–95% band", fontsize=9)
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig("fig_21_randstorm.png", dpi=130); plt.show()

In [ ]:
report = {"generated_at": dt.datetime.now(dt.timezone.utc).isoformat(), "bq_usd": round(SPENT["usd"], 3),
          "windows": WINDOWS, "pubform_counts": chk.astype(str).to_dict(orient="records"),
          "sizes_main_window": sz.loc["main"].reset_index().astype(str).to_dict(orient="records"),
          "pc": s[s.event.isin(["randstorm", "ftx"])].astype(str).to_dict(orient="records"),
          "pooled_main": ps.astype(str).to_dict(orient="records")}
with open("build_report_21.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)
try:
    from google.colab import files
    for fn in ["build_report_21.json", "est_21_effects.csv", "est_21_pooled.csv", "pubform_by_month.csv",
               "fig_21_pubform.png", "fig_21_randstorm.png"]:
        if os.path.exists(fn): files.download(fn)
except Exception:
    print("Report saved to build_report_21.json")